# Step 1：（加载环境与选择模型）
- 本单元会定位项目根目录，并读取 `config.yaml`。
- 这里加入可交互的 provider/model 选择面板，方便你直接切换模型做测试。
- 下面的代码单元会展示下拉框、输入框和运行按钮。

In [ ]:
# 分块1：导入基础模块
from pathlib import Path
import os
import yaml
from dotenv import load_dotenv
from IPython.display import display, clear_output

# 分块2：定位项目根目录并加载环境变量
PROJECT_ROOT = Path.cwd()
load_dotenv(PROJECT_ROOT / '.env')

# 分块3：读取配置并构建交互式模型选择面板
config = yaml.safe_load((PROJECT_ROOT / 'config.yaml').read_text(encoding='utf-8'))

provider_models = {
    'openrouter': [
        'moonshotai/kimi-k2.5',
        'qwen/qwen-vl-plus',
        'tencent/hy3-preview:free',
    ],
    'deepseek': [
        'deepseek-chat',
    ],
}

initial_provider = config.get('active_provider', 'openrouter')
initial_model = 'tencent/hy3-preview:free' if initial_provider == 'openrouter' else config['providers'][initial_provider]['model']

try:
    import ipywidgets as widgets
    from core.runtime import build_agent

    provider_dropdown = widgets.Dropdown(
        options=['openrouter', 'deepseek'],
        value=initial_provider,
        description='Provider:',
        layout=widgets.Layout(width='420px'),
    )
    model_dropdown = widgets.Dropdown(
        options=provider_models[provider_dropdown.value],
        value=initial_model,
        description='Model:',
        layout=widgets.Layout(width='420px'),
    )
    prompt_text = widgets.Textarea(
        value='你好,我叫小明，请用当前模型做一次简短自我介绍。',
        description='Prompt:',
        layout=widgets.Layout(width='760px', height='110px'),
    )
    run_button = widgets.Button(description='Run Agent', button_style='primary')
    output_area = widgets.Output()

    def sync_models(change=None):
        options = provider_models[provider_dropdown.value]
        model_dropdown.options = options
        if provider_dropdown.value == 'openrouter':
            model_dropdown.value = 'tencent/hy3-preview:free'
        elif model_dropdown.value not in options:
            model_dropdown.value = options[0]

    def on_run_clicked(_):
        with output_area:
            clear_output()
            try:
                active_provider = provider_dropdown.value
                active_model = model_dropdown.value
                print('active_provider =', active_provider)
                print('active_model =', active_model)
                agent = build_agent(config, provider_override=active_provider, model_override=active_model)
                reply = agent.run(prompt_text.value)
                print('\n=== Agent Reply ===')
                print(reply)
            except Exception as exc:
                print('运行失败：', exc)

    provider_dropdown.observe(sync_models, names='value')
    run_button.on_click(on_run_clicked)

    sync_models()
    display(provider_dropdown, model_dropdown, prompt_text, run_button, output_area)

except Exception as exc:
    print('ipywidgets 不可用，降级为文本方式。原因：', exc)
    print('active_provider =', config['active_provider'])
    print('default model(openrouter) =', initial_model)
    print('default model(deepseek) =', config['providers']['deepseek']['model'])
    print('你也可以在后续单元手动调用 core.runtime.build_agent(...) 进行测试。')

Dropdown(description='Provider:', layout=Layout(width='420px'), options=('openrouter', 'deepseek'), value='ope…

Dropdown(description='Model:', index=2, layout=Layout(width='420px'), options=('moonshotai/kimi-k2.5', 'qwen/q…

Textarea(value='你好,我叫小明，请用当前模型做一次简短自我介绍。', description='Prompt:', layout=Layout(height='110px', width='760px')…

Button(button_style='primary', description='Run Agent', style=ButtonStyle())

Output()

# Step 2：（初始化示例数据库）
- 本单元调用项目内的 `seed_sample_db.py`。
- 完成 `data/sample.db` 的建表与 10 条样例数据写入。

In [2]:
# 分块1：执行数据库种子脚本
from data.seed_sample_db import main as seed_db

# 分块2：初始化并校验文件存在
seed_db()
print('db exists =', (PROJECT_ROOT / 'data' / 'sample.db').exists())

Seeded database at data\sample.db with 10 rows
db exists = True


# Step 3：（扫描并展示 Skills 目录）
- 本单元复用 `SkillLoader` 扫描 `skills/`。
- 输出轻量目录文本，验证即插即用机制。

In [11]:
# 分块1：导入并初始化 SkillLoader
from core.skills import SkillLoader

loader = SkillLoader(PROJECT_ROOT / 'skills')
loader.scan()

# 分块2：打印 catalog 文本
print(loader.get_catalog_text())

Available skills (use activate_skill to load details):
- hello-world: Greet the user by name and demonstrate the skill activation mechanism. Use when user says hello, hi, 你好, or asks to test skills.
- sqlite-sample: Query the sample SQLite employee database for analytics and lookup tasks. Use for salary, department, and employee queries.


# Step 4：（演示四个内置工具的注册与调用）
- 本单元创建 `ToolRegistry` 并注册 read/write/bash/activate_skill。
- 依次执行写文件、读文件、激活技能、执行命令。

In [4]:
# 分块1：导入核心组件
import logging
from core.tools import ToolRegistry
from tools_builtin.file_ops import create_file_handlers
from tools_builtin.shell import create_bash_handler
from tools_builtin.skill_ops import create_activate_skill_handler

# 分块2：注册四个工具
registry = ToolRegistry()
read_handler, write_handler = create_file_handlers(PROJECT_ROOT)
bash_handler = create_bash_handler(PROJECT_ROOT, logging.getLogger('nb.bash'))
activate_handler = create_activate_skill_handler(loader)

registry.register('read', 'read file', {'type': 'object', 'properties': {'path': {'type': 'string'}}, 'required': ['path']}, read_handler)
registry.register('write', 'write file', {'type': 'object', 'properties': {'path': {'type': 'string'}, 'content': {'type': 'string'}}, 'required': ['path', 'content']}, write_handler)
registry.register('bash', 'run shell', {'type': 'object', 'properties': {'command': {'type': 'string'}}, 'required': ['command']}, bash_handler)
registry.register('activate_skill', 'activate skill', {'type': 'object', 'properties': {'name': {'type': 'string'}}, 'required': ['name']}, activate_handler)

# 分块3：执行工具调用
print(registry.execute('write', {'path': 'workspace/notebook-demo.txt', 'content': 'hello from notebook'}))
print(registry.execute('read', {'path': 'workspace/notebook-demo.txt'}))
print(registry.execute('activate_skill', {'name': 'hello-world'})[:120])
print(registry.execute('bash', {'command': 'echo notebook_bash_ok'}))

Wrote 19 chars to D:\Agent\mini-agent\workspace\notebook-demo.txt
hello from notebook
# Hello World Skill

This skill verifies that skill activation and bash execution work correctly.

## How To Use

1. Ext
[exit_code=0]
STDOUT:
notebook_bash_ok

STDERR:



# Step 5：（执行 sqlite-sample Skill 脚本）
- 本单元通过子进程调用 `query.py --sql`。
- 展示 JSON 查询结果，验证 DB-as-skill 模式。

In [6]:
# 分块1：调用 sqlite skill 脚本
import subprocess

cmd = ['python', 'skills/sqlite-sample/scripts/query.py', '--sql', 'SELECT name, salary FROM employees ORDER BY salary DESC LIMIT 3']
res = subprocess.run(cmd, cwd=PROJECT_ROOT, capture_output=True, text=True)

# 分块2：打印标准输出与错误码
print('exit_code =', res.returncode)
print(res.stdout)
if res.stderr:
    print('stderr =', res.stderr)

exit_code = 0
{
  "count": 3,
  "rows": [
    {
      "name": "Alice",
      "salary": 165000.0
    },
    {
      "name": "Bob",
      "salary": 148000.0
    },
    {
      "name": "Carol",
      "salary": 132000.0
    }
  ]
}



# Step 6：（Notebook 内嵌 CLI 交互演示）
- 本单元会在 Notebook 内直接提供 CLI 风格交互，不需要再开外部终端。
- 你可以连续输入问题，实时看到工具调用与 Agent 回复。
- 输入 `exit` 或 `quit` 可在当前面板中结束会话。

In [ ]:
# 分块1：导入交互组件并确定当前模型
import ipywidgets as widgets
from IPython.display import display, clear_output
from core.runtime import build_agent

active_provider = globals().get('provider_dropdown').value if 'provider_dropdown' in globals() else config.get('active_provider', 'openrouter')
active_model = globals().get('model_dropdown').value if 'model_dropdown' in globals() else config['providers'][active_provider]['model']

# 分块2：构建当前模型对应的 Agent
cli_agent = build_agent(config, provider_override=active_provider, model_override=active_model)

# 分块3：创建 Notebook 内嵌 CLI 交互面板
cli_input = widgets.Textarea(
    value='',
    placeholder='在这里输入问题，然后点发送。输入 exit 或 quit 结束。',
    description='You:',
    layout=widgets.Layout(width='900px', height='90px'),
)
send_btn = widgets.Button(description='发送', button_style='primary')
clear_btn = widgets.Button(description='清屏', button_style='')
session_info = widgets.HTML(value=f"<b>Provider:</b> {active_provider} &nbsp;&nbsp; <b>Model:</b> {active_model}")
cli_output = widgets.Output(layout={'border': '1px solid #ddd', 'padding': '8px'})

with cli_output:
    print('Mini Agent notebook-cli 已启动。输入 exit 或 quit 可结束会话。')

# 分块4：定义交互逻辑（显示工具调用 + 最终回复）
def _handle_send(_):
    user_text = cli_input.value.strip()
    if not user_text:
        D:/anaconda/envs/mywork/python.exe main.py webui        D:/anaconda/envs/mywork/python.exe main.py webui        return

    with cli_output:
        print(f'You> {user_text}')

    if user_text.lower() in {'exit', 'quit'}:
        with cli_output:
            print('Agent> 会话已结束。你可以重新运行本单元开始新会话。')
        send_btn.disabled = True
        cli_input.disabled = True
        return

    steps = []
    try:
        reply = cli_agent.run(user_text, on_step=lambda s: steps.append(s))
        with cli_output:
            for step in steps:
                print(f"[tool] {step.get('name')} {step.get('args', {})}")
            print(f'Agent> {reply}')
    except Exception as exc:
        with cli_output:
            print(f'Agent> [error] {exc}')

    cli_input.value = ''


def _handle_clear(_):
    with cli_output:
        clear_output()
        print('Mini Agent notebook-cli 已清屏，继续输入即可。')


send_btn.on_click(_handle_send)
clear_btn.on_click(_handle_clear)

display(session_info, cli_input, widgets.HBox([send_btn, clear_btn]), cli_output)

HTML(value='<b>Provider:</b> openrouter &nbsp;&nbsp; <b>Model:</b> tencent/hy3-preview:free')

Textarea(value='', description='You:', layout=Layout(height='90px', width='900px'), placeholder='在这里输入问题，然后点发送…

Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…